# The Dynamics

In this guide you will learn how a cooperation game evolves and how pyCoop simulates it.
You can then run a game on any graph and keep the full record of what happened, which the next guide measures.

A cooperation game is a piecewise-deterministic random process, in which ballistic drift is broken up by decisions made at random times.
Between decisions the positions depend only on the elapsed time and the headings, and the flip rule reads the state only at a decision.
The process therefore reduces to an embedded chain, in which one uniformly random player acts at each event and the events are spaced exponentially in time.
The chain reproduces the full process, so pyCoop runs it directly, with no event queue.
The model is the Agreement-Position Emigration model of Conrad and Tabor (2024), who give the full analysis.
We follow the notation of its socio-physics recasting.

## The update sequence

Let there be $N$ agents on a connected graph with adjacency matrix $A_{ij}$ and no self-loops.
Each agent carries two degrees of freedom, a continuous *action* $\sigma_i \in [-1, 1]$ and a binary *opinion* $q_i \in \{-1, +1\}$.
Think of a person who believes something and is doing something about it; what they believe and what they do need not agree.

On each step we draw two random variables,

$$
\tau \sim \mathrm{Exp}\!\left[\tfrac{N}{T}\right],
\qquad
j \sim \mathcal{U}_N,
$$

where $\tau$ is how long we wait and $j$ is who gets to act.
Each action drifts toward its opinion and stops at the walls,

$$
\sigma_i \mapsto \min(\max(\sigma_i + q_i\,\tau,\, -1),\, 1),
$$

and player $j$ alone then compares their action with their neighborhood's.
The *local mean action* is

$$
\bar\sigma_j = \frac{1}{d_j + 1}\Big(\sigma_j + \sum_k A_{jk}\,\sigma_k\Big)
$$

where the average counts the player alongside their $d_j$ neighbors, since an agent listens to the crowd and weighs themselves too.
Including the player gives the prefactor $1/(d_j+1)$; an average over the neighbors alone would have $1/d_j$.
The opinion flips, $q_j \mapsto -q_j$, iff

$$
-u < \bar\sigma_j < u
\quad\text{and}\quad
|\sigma_j| > l.
$$

The first condition says the neighborhood is still undecided.
The second says the player has drifted far enough out to reconsider.
The only dimensionless timescale in the problem is $T$.
The pair $(l, u)$ - with $\Delta = u - l > 0$ - marks out the change-of-heart regions $[-u, -l] \cup [l, u]$, where flips can happen.

## Running a game

A [`Game`](#pycoop.dynamics.Game) pairs a graph with the three parameters:

In [ ]:
# Install pyCoop first when this notebook runs somewhere it is not installed,
# such as Google Colab. Where pyCoop is already installed this does nothing.
try:
    import pycoop
except ImportError:
    %pip install --quiet git+https://github.com/EternalTime/pyCoop.git

In [ ]:
import pycoop as pc

game = pc.Game(pc.erg_connected(20, 0.1, rng=0), u=0.5, l=0.2, T=0.1)
traj = game.run(rng=1, max_steps=200_000)

The method `run` takes initial conditions (`sigma0` and `q0`, drawn uniformly by default), a maximum number of steps, a maximum time (`max_time`), and a seed.
It stops early once the state is *absorbed*.
In an absorbed state each action sits pinned at its opinion's wall and each agreement window is shut, so no flip can happen again.
Consensus is the absorbing corner where all the signs agree, and the other absorbing corners hold the fraught and obdurate states.

## The trajectory

The run comes back as a [`Trajectory`](#pycoop.dynamics.Trajectory) holding the event times, the actions and opinions after each event, the player who acted, and whether they flipped.
All the randomness lives in the iid sequence $(\tau_k, j_k)$, so the trajectory is a complete record of the run.
The simulator only records the trajectory, and each observable in the next guide is computed from it after the run.

## Changing the clock

Exponential waits reproduce the model above.
To try other waits, use `step_law`.
Pass a float and each wait becomes a fixed step $h$; pass a function and the waits follow any law you like:

In [ ]:
game = pc.Game(pc.complete(64), u=0.5, l=0.2, T=2.0, step_law=0.03125)

The fixed-step variant is a different model.
Its positions live on a lattice of spacing $h$, and the dispersion of step sizes enters only through the resolution near the thresholds.
Whether the difference matters is an empirical question.
The answer decides whether the critical behavior comes from the dynamics or from the clock.
If the behavior is the same under every clock law, the value of $T$ does not change it.
Find out by running the same game under both clocks.